# ML Model Notebook: XGBoost Financial Direction Model

This notebook follows the full production lifecycle for the ML pipeline: data ingestion, cleaning, feature engineering, training, evaluation, and artifact export for FastAPI inference.

Target artifacts:
- `xgb_model.pkl`
- `scaler.pkl`

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'backend').exists() and (REPO_ROOT.parent / 'backend').exists():
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / 'data' / 'raw'
MODEL_DIR = REPO_ROOT / 'backend' / 'models'
RAW_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def load_market_data(symbol='AAPL', period='5y'):
    csv_path = RAW_DIR / f'{symbol.lower()}_raw.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df['date'] = pd.to_datetime(df['date'])
        return df

    try:
        df = yf.download(symbol, period=period, auto_adjust=False, progress=False).reset_index()
        df.columns = [c.lower().replace(' ', '_') for c in df.columns]
        df = df.rename(columns={'adj_close': 'adj_close'})
        df = df[['date', 'open', 'high', 'low', 'close', 'volume']]
        df.to_csv(csv_path, index=False)
        return df
    except Exception:
        dates = pd.date_range(end=pd.Timestamp.today(), periods=500, freq='B')
        rng = np.random.default_rng(42)
        trend = np.linspace(100, 180, len(dates))
        noise = rng.normal(0, 2, len(dates)).cumsum()
        close = trend + noise
        df = pd.DataFrame({
            'date': dates,
            'open': close + rng.normal(0, 1, len(dates)),
            'high': close + rng.normal(1, 1, len(dates)),
            'low': close - rng.normal(1, 1, len(dates)),
            'close': close,
            'volume': rng.integers(900000, 2200000, len(dates)),
        })
        df.to_csv(csv_path, index=False)
        return df

In [ ]:
df = load_market_data('AAPL')
display(df.head())
print('shape:', df.shape)
display(df.info())
display(df.describe(include='all').transpose())

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

In [ ]:
def cap_outliers(series: pd.Series) -> pd.Series:
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper)

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    data = data.drop_duplicates().sort_values('date').reset_index(drop=True)
    data['date'] = pd.to_datetime(data['date'])
    data[['open', 'high', 'low', 'close', 'volume']] = data[['open', 'high', 'low', 'close', 'volume']].apply(pd.to_numeric, errors='coerce')
    data['close'] = data['close'].fillna(data['close'].median())
    data['volume'] = data['volume'].fillna(data['volume'].median())
    for col in ['open', 'high', 'low', 'close', 'volume']:
        data[col] = data[col].fillna(data[col].median())
        data[col] = cap_outliers(data[col])

    data['return_1'] = data['close'].pct_change().fillna(0)
    data['return_5'] = data['close'].pct_change(5).fillna(0)
    data['sma_10'] = data['close'].rolling(10).mean()
    data['sma_20'] = data['close'].rolling(20).mean()
    data['ema_10'] = data['close'].ewm(span=10, adjust=False).mean()
    data['volatility_10'] = data['close'].pct_change().rolling(10).std().fillna(0)
    data['volume_zscore'] = (data['volume'] - data['volume'].rolling(20).mean()) / data['volume'].rolling(20).std()
    data['volume_zscore'] = data['volume_zscore'].replace([np.inf, -np.inf], np.nan).fillna(0)
    data['target'] = (data['close'].shift(-1) > data['close']).astype(int)
    data = data.dropna().reset_index(drop=True)
    return data

dataset = build_features(df)
display(dataset.head())
print('final shape:', dataset.shape)
assert not dataset.isna().any().any()

In [ ]:
feature_columns = ['return_1', 'return_5', 'sma_10', 'sma_20', 'ema_10', 'volatility_10', 'volume_zscore']
X = dataset[feature_columns]
y = dataset['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
)
xgb_model.fit(X_train_scaled, y_train)
y_pred = xgb_model.predict(X_test_scaled)
y_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
print({'accuracy': accuracy, 'precision': precision, 'recall': recall, 'roc_auc': auc})

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title('ROC Curve')
plt.tight_layout()
plt.show()

feature_importance = pd.Series(xgb_model.feature_importances_, index=feature_columns).sort_values(ascending=False)
feature_importance.plot(kind='bar', color='#2563eb', title='XGBoost Feature Importance')
plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    'actual': y_test.to_numpy(),
    'predicted': y_pred,
    'prob_up': y_prob,
}).reset_index(drop=True)
display(comparison.head(20))

In [ ]:
artifact_dir = MODEL_DIR
joblib.dump(xgb_model, artifact_dir / 'xgb_model.pkl')
joblib.dump(scaler, artifact_dir / 'scaler.pkl')
print('Saved:', artifact_dir / 'xgb_model.pkl')
print('Saved:', artifact_dir / 'scaler.pkl')

sample_features = scaler.transform(X_test.iloc[:1])
print('Sample prediction probability:', xgb_model.predict_proba(sample_features)[0, 1])

## Backend Integration Notes

The FastAPI backend loads the exported XGBoost model and scaler from `backend/models/`. The notebook does not contain any inference server logic. It only prepares the artifact files used by the API layer.